In [0]:
import pandas as pd


In [0]:
df_final=spark.read.table("post_renewal_churn.cleaned_dataset.final_dataset").toPandas()


In [0]:
df_final["prospect_outcome"].value_counts()

In [0]:
df_final.shape
df_final.head()
df_final.info()

In [0]:
df_final['prospect_outcome'].value_counts()
df_final['prospect_outcome'].value_counts(normalize=True) * 100

In [0]:
num_cols = df_final.select_dtypes(include=['int64', 'float64']).columns

df_final[num_cols].describe()

In [0]:
cat_cols = df_final.select_dtypes(include=['object']).columns

for col in cat_cols:
    print(f"\n{col}")
    print(df_final[col].value_counts().head())

In [0]:
pd.crosstab(df_final['tenure_group'], df_final['prospect_outcome'], normalize='index') * 100

In [0]:
df_final['label'] = df_final['prospect_outcome'].map({'Won': 0, 'Churned': 1})

In [0]:
corr = df_final[num_cols.tolist() + ['label']].corr()

corr['label'].sort_values(ascending=False)

In [0]:
df_final['Renewal_Month'] = pd.to_datetime(df_final['Renewal_Month'], dayfirst=True, errors='coerce')

In [0]:
df_final['Year'] = df_final['Renewal_Month'].dt.year
df_final['Month'] = df_final['Renewal_Month'].dt.month

In [0]:
churn_data = df_final[df_final['prospect_outcome'] == 'Churned']

year_month_churn = churn_data.groupby(['Year', 'Month']).size().reset_index(name='Churn_Count')

year_month_churn

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

pivot_table = year_month_churn.pivot(index='Month', columns='Year', values='Churn_Count')

pivot_table.plot(kind='bar', figsize=(10,6))

plt.title("Year-wise Monthly Churn Count")
plt.ylabel("Churn Count")
plt.xlabel("Month")
plt.show()

In [0]:
plt.figure(figsize=(10,6))
sns.heatmap(pivot_table, annot=True, fmt='g', cmap='coolwarm')

plt.title("Churn Heatmap (Year vs Month)")
plt.show()

In [0]:
pd.crosstab(df_final['Call_Direction'], df_final['prospect_outcome'], normalize='index') * 100

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in ['Tenure_Years', 'total_renewal_score_new']:
    sns.histplot(data=df_final, x=col, hue='prospect_outcome', kde=True)
    plt.show()

In [0]:
band_churn = pd.crosstab(
    df_final['Band'],
    df_final['prospect_outcome'],
    normalize='index'
) * 100

band_churn = band_churn.sort_values(by='Churned', ascending=False)

band_churn.plot(kind='bar', figsize=(10,5))
plt.title("Churn % by Band (Sorted)")
plt.ylabel("Percentage")
plt.xticks(rotation=45)
plt.show()

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

df_model = df_final.copy()

# Convert categorical → numeric
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = LabelEncoder().fit_transform(df_model[col].astype(str))

X = df_model.drop(['prospect_outcome', 'Co_Ref'], axis=1)
y = df_model['label']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

importances = pd.Series(model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(15)

top_features.plot(kind='barh', figsize=(10,6))
plt.title("Top Features Affecting Churn")
plt.show()

In [0]:
important_cols = ['Tenure_Years', 'Total_Amount', 'Gross']

for col in important_cols:
    sns.boxplot(x='prospect_outcome', y=col, data=df_final)
    plt.title(f"{col} vs Churn")
    plt.show()

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select only numeric columns
num_cols = [
    'Tenure_Years', 'Total_Amount', 'Gross', 'Membership_Net',
    'total_renewal_score_new', 'status_scores',
    'sustainability_score', 'anchoring_score',
    'Current_Anchorings', 'days_to_close'
]

# Add target
df_corr = df_final[num_cols].copy()
df_corr['label'] = df_final['prospect_outcome'].map({'Won': 0, 'Churned': 1})

# Correlation matrix
corr = df_corr.corr()

# Plot
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Correlation Matrix (Numerical Features)")
plt.show()

In [0]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(k-1, r-1))))

cat_cols = [
    'Band', 'Call_Direction', 'Payment_Method',
    'Serious_Complaint', 'Other_Complaint',
    'Discount_or_Waiver_Requested',
    'Renewal_Impact_Due_to_Price_Increase'
]

for col in cat_cols:
    print(col, cramers_v(df_final[col], df_final['prospect_outcome']))

In [0]:
pd.crosstab(
    df_final['Payment_Method'],
    df_final['prospect_outcome']
)

# Unnecessary features drop


In [0]:
# Columns to drop
drop_cols = [
    #  IDs (no predictive value)
    'Co_Ref', 'index', 'Call_ID',
    
    # Leakage (future / target-dependent)
    'Closed_Date', 'Call_Date', 'Prospect_Renewal_Date',
    'Payment_Method', 'days_to_close',
    
    # Constant column
    'Analysed_Call',
    
    # Multicollinearity (duplicate info)
    'Gross', 'Membership_Net',
    
    #  Duplicate signal
    'status_scores',
    
    # Weak / no impact features
    'Serious_Complaint', 'Other_Complaint', 'Discount_or_Waiver_Requested'
]

# Drop them
df_model = df_final.drop(columns=drop_cols)

# Check result
print("Remaining columns:", df_model.shape[1])
df_model.head()

# select only the necessary features from the dataset


In [0]:
# ===============================
# 1. Select ONLY necessary features
# ===============================

num_features = [
    'Tenure_Years',
    'Total_Amount',
    'total_renewal_score_new',
    'sustainability_score',
    'anchoring_score',
    'Current_Anchorings'
]

cat_features = [
    'Call_Direction',
    'Renewal_Impact_Due_to_Price_Increase',
    'Band'
]

features = num_features + cat_features

# Create final dataset
df_selected = df_final[features + ['prospect_outcome']].copy()

# ===============================
# 2. Convert target to numeric (optional for later)
# ===============================

df_selected['prospect_outcome'] = df_selected['prospect_outcome'].map({
    'Won': 0,
    'Churned': 1
})

# ===============================
# 3. Check result
# ===============================

print("Final shape:", df_selected.shape)
df_selected.head()

In [0]:
spark.createDataFrame(df_selected).write.mode('overwrite').saveAsTable("post_renewal_churn.cleaned_dataset.features_table")

In [0]:
display(df_selected)